# Hybrid Retrieval: Combining BM25 and Semantic Search

Combine keyword-based (BM25) and semantic (embedding) retrieval for better results.

## Output
- **src/hybrid.py** - HybridRetriever class (auto-generated)
- Verified hybrid retrieval working
- Comparison of BM25 vs Semantic vs Hybrid
- Ready for RAG pipeline in 07_rag_pipeline.ipynb

## Workflow
1. Check prerequisites (both indexes exist)
2. Load BM25 index
3. Load Semantic index
4. Auto-generate src/hybrid.py
5. Test on sample queries
6. Compare all three methods
7. Verify hybrid works

## 1. Setup and Prerequisites

Initializes paths and checks that both BM25 and Semantic indexes exist from phases 1-2.

**Output:** Project root and prerequisite file status

In [1]:
from pathlib import Path
import sys

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

corpus_file = project_root / "data" / "processed" / "corpus.pkl"
bm25_index_file = project_root / "data" / "processed" / "bm25_index.pkl"
semantic_index_file = project_root / "data" / "processed" / "semantic_index" / "faiss_index"
books_data_file = project_root / "data" / "processed" / "books_sample.parquet"

print(f"Project root: {project_root}")
print()
print("Prerequisites check:")
print(f"  corpus.pkl exists: {corpus_file.exists()}")
print(f"  bm25_index.pkl exists: {bm25_index_file.exists()}")
print(f"  semantic_index/faiss_index exists: {semantic_index_file.exists()}")
print(f"  books_sample.parquet exists: {books_data_file.exists()}")

missing = [f for f in [corpus_file, bm25_index_file, semantic_index_file, books_data_file] if not f.exists()]

if missing:
    print(f"\nERROR: Missing files. Run 03_bm25 and 04_semantic notebooks first.")
else:
    print("\nAll prerequisites ready.")

Project root: /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki

Prerequisites check:
  corpus.pkl exists: True
  bm25_index.pkl exists: True
  semantic_index/faiss_index exists: True
  books_sample.parquet exists: True

All prerequisites ready.


## 2. Load BM25 Index

Loads the BM25 keyword search index from phase 1.

**Output:** BM25 index confirmation

In [2]:
import pickle
import importlib

# Force reimport
if 'src.bm25' in sys.modules:
    importlib.reload(sys.modules['src.bm25'])

from src.bm25 import BM25Retriever

print("Loading BM25 index...")

bm25_retriever = BM25Retriever()
bm25_retriever.load(str(bm25_index_file))

with open(corpus_file, "rb") as f:
    corpus = pickle.load(f)

bm25_retriever.corpus = corpus

print(f"BM25 index loaded")
print(f"Corpus: {len(corpus):,} documents")

Loading BM25 index...
BM25 index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/bm25_index.pkl
BM25 index loaded
Corpus: 91,850 documents


## 3. Load Semantic Index

Loads the semantic embedding index from phase 2.

**Output:** Semantic index confirmation

In [3]:
# Force reimport
if 'src.semantic_retriever' in sys.modules:
    importlib.reload(sys.modules['src.semantic_retriever'])

from src.semantic_retriever import SemanticRetriever

print("Loading Semantic index...")

semantic_retriever = SemanticRetriever()
semantic_retriever.load(str(semantic_index_file))
semantic_retriever.corpus = corpus

print(f"Semantic index loaded")

Loading Semantic index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index loaded from /Users/esteki/Desktop/MDS/575/test/DSCI_575_project_jchuang_esteki/data/processed/semantic_index/faiss_index
Semantic index loaded


## 4. Auto-Generate src/hybrid.py

Converts the round-robin interleaving strategy into a reusable `HybridRetriever` class and writes it to `src/hybrid.py`. Follows the same auto-generation pattern as `src/bm25.py` (notebook 03) and `src/semantic_retriever.py` (notebook 04).

**Output:** `src/hybrid.py` file created

In [4]:
hybrid_code = '''from typing import List, Tuple


class HybridRetriever:
    """Hybrid retriever combining BM25 and semantic embedding search.

    Uses round-robin interleaving to merge ranked results from both retrievers,
    returning documents with source attribution (BM25/Semantic/BM25 + Semantic).
    """

    def __init__(self, bm25_retriever, semantic_retriever):
        self.bm25 = bm25_retriever
        self.semantic = semantic_retriever

    def search(self, query: str, top_k: int = 5) -> List[Tuple[int, str]]:
        """Search using both retrievers and merge via round-robin interleaving.

        Alternates picking from each method\'s ranked list, deduplicating as it goes,
        until top-k results are collected. Returns a list of (doc_id, source) tuples
        where source is \'BM25\', \'Semantic\', or \'BM25 + Semantic\'.
        """
        bm25_results = self.bm25.search(query, top_k)
        semantic_results = self.semantic.search(query, top_k)

        bm25_ids = [idx for idx, _ in bm25_results]
        semantic_ids = [idx for idx, _ in semantic_results]
        overlap = set(bm25_ids) & set(semantic_ids)

        # Round-robin interleave: alternate picking from BM25 and semantic so both
        # methods are fairly represented in the final top_k. Without this, Python\'s
        # dict insertion order would mean BM25 results always fill the top slots first,
        # leaving no room for semantic-only results.
        combined = []
        seen = set()
        for bm25_id, semantic_id in zip(bm25_ids, semantic_ids):
            for doc_id, id_list in ((bm25_id, bm25_ids), (semantic_id, semantic_ids)):
                if doc_id not in seen:
                    source = (
                        "BM25 + Semantic" if doc_id in overlap
                        else ("BM25" if id_list is bm25_ids else "Semantic")
                    )
                    combined.append((doc_id, source))
                    seen.add(doc_id)
                if len(combined) == top_k:
                    return combined

        return combined
'''

hybrid_file = project_root / "src" / "hybrid.py"
with open(hybrid_file, 'w') as f:
    f.write(hybrid_code)

print("Created src/hybrid.py")


Created src/hybrid.py


## 5. Test Hybrid Retrieval

Tests hybrid search on sample queries. Shows which results come from which method.

**Output:** Hybrid search results for each query

In [5]:
import importlib
import pandas as pd

if 'src.hybrid' in sys.modules:
    importlib.reload(sys.modules['src.hybrid'])

from src.hybrid import HybridRetriever

hybrid_retriever = HybridRetriever(bm25_retriever, semantic_retriever)

print("Testing Hybrid Retrieval")
print("=" * 80)

books_data = pd.read_parquet(books_data_file)

test_queries = ["python book", "mystery novel", "machine learning"]

for query in test_queries:
    hybrid_results = hybrid_retriever.search(query, top_k=5)
    print(f"\nQuery: '{query}'")
    print(f"Results: {len(hybrid_results)} documents")
    for rank, (doc_id, source) in enumerate(hybrid_results, 1):
        book_title = books_data.iloc[doc_id]['product_title']
        rating = books_data.iloc[doc_id]['rating']
        print(f"  {rank}. [{source}] Doc #{doc_id} - {book_title} ({rating}/5)")


Testing Hybrid Retrieval

Query: 'python book'
Results: 5 documents
  1. [BM25] Doc #88231 - Python: Programming For Beginners: Learn The Fundamentals of Python in 7 Days (5.0/5)
  2. [Semantic] Doc #55956 - Python Programming for Advanced: Learn the Fundamentals of Python in 7 Days (1.0/5)
  3. [BM25] Doc #7535 - Effective Python: 59 Specific Ways to Write Better Python (Effective Software Development Series) (5.0/5)
  4. [Semantic] Doc #50152 - Programming Python (2.0/5)
  5. [BM25] Doc #65874 - Python for Everybody: Exploring Data in Python 3 (5.0/5)

Query: 'mystery novel'
Results: 5 documents
  1. [BM25] Doc #39571 - The Woman in the Window: A Novel (5.0/5)
  2. [Semantic] Doc #42383 - And Then There Were None (5.0/5)
  3. [BM25] Doc #63216 - One Little Wish: a romantic suspense novel (The Little Things Mystery Series) (5.0/5)
  4. [Semantic] Doc #23725 - Mystery: An Alex Delaware Novel (3.0/5)
  5. [BM25] Doc #47298 - The Last Lie Told (Finley O’Sullivan Book 1) (5.0/5)

Query: '

## 6. Compare: BM25 vs Semantic vs Hybrid

Shows differences between methods on same query. Demonstrates why hybrid helps.

The query "good book recommendation" is intentionally chosen to maximise divergence: BM25 matches on the literal word "recommendation" (surfacing titles like "A Book Club Recommend"), while semantic search understands the intent and returns completely different results. Zero overlap between the two methods is the whole point, as it shows each method's distinct strengths.

**Output:** Side-by-side comparison showing unique results from each method

In [6]:
print("Comparison: BM25 vs Semantic vs Hybrid")
print("=" * 80)

test_query = "good book recommendation"
print(f"\nQuery: '{test_query}'")

# Get results from each method
bm25_results = bm25_retriever.search(test_query, top_k=5)
semantic_results = semantic_retriever.search(test_query, top_k=5)
hybrid_results = hybrid_retriever.search(test_query, top_k=5)

bm25_ids = set(idx for idx, _ in bm25_results)
semantic_ids = set(idx for idx, _ in semantic_results)

print(f"\nBM25 (keyword): {len(bm25_ids)} unique results")
for idx in bm25_ids:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nSemantic (embedding): {len(semantic_ids)} unique results")
for idx in semantic_ids:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nHybrid (combined): {len(hybrid_results)} results")
for doc_id, source in hybrid_results:
    print(f"  [{source}] {books_data.iloc[doc_id]['product_title'][:50]}")
print(f"  From BM25 only: {sum(1 for _, src in hybrid_results if src == 'BM25')}")
print(f"  From Semantic only: {sum(1 for _, src in hybrid_results if src == 'Semantic')}")
print(f"  From both: {sum(1 for _, src in hybrid_results if src == 'BM25 + Semantic')}")


Comparison: BM25 vs Semantic vs Hybrid

Query: 'good book recommendation'

BM25 (keyword): 5 unique results
  - Memoirs of a Not Altogether Shy Pornographer
  - It All Comes Back to You: A Book Club Recommendati
  - Photo Composition Mastery! (On Target Photo Traini
  - Noble Man (Jake Noble Series Book 1)
  - Dinosaur Dance!

Semantic (embedding): 5 unique results
  - Phoenix Island
  - Everything, Everything
  - Near Future
  - In Times Like These
  - Write Better Essays in 20 Minutes a Day

Hybrid (combined): 5 results
  [BM25] Noble Man (Jake Noble Series Book 1)
  [Semantic] Write Better Essays in 20 Minutes a Day
  [BM25] Dinosaur Dance!
  [Semantic] Phoenix Island
  [BM25] Memoirs of a Not Altogether Shy Pornographer
  From BM25 only: 3
  From Semantic only: 2
  From both: 0


### Example 2

This next example shows where a hybrid approach can be useful. We get results from BM25, semantic search, and also their overlapping union.

In [7]:
test_query = "programming book"
print(f"Query: '{test_query}'")

bm25_results = bm25_retriever.search(test_query, top_k=5)
semantic_results = semantic_retriever.search(test_query, top_k=5)
hybrid_results = hybrid_retriever.search(test_query, top_k=5)

bm25_ids = set(idx for idx, _ in bm25_results)
semantic_ids = set(idx for idx, _ in semantic_results)

print(f"\nBM25 (keyword): {len(bm25_ids)} unique results")
for idx in bm25_ids:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nSemantic (embedding): {len(semantic_ids)} unique results")
for idx in semantic_ids:
    print(f"  - {books_data.iloc[idx]['product_title'][:50]}")

print(f"\nHybrid (combined): {len(hybrid_results)} results")
for doc_id, source in hybrid_results:
    print(f"  [{source}] {books_data.iloc[doc_id]['product_title'][:50]}")
print(f"  From BM25 only: {sum(1 for _, src in hybrid_results if src == 'BM25')}")
print(f"  From Semantic only: {sum(1 for _, src in hybrid_results if src == 'Semantic')}")
print(f"  From both: {sum(1 for _, src in hybrid_results if src == 'BM25 + Semantic')}")


Query: 'programming book'

BM25 (keyword): 5 unique results
  - Hands-On Object-Oriented Programming with C#: Buil
  - Concepts of Programming Languages
  - The Art of LEGO MINDSTORMS EV3 Programming (Full C
  - Beginning C for Arduino, Second Edition: Learn C P
  - Parallel and Concurrent Programming in Haskell: Te

Semantic (embedding): 5 unique results
  - The Science of Programming (Monographs in Computer
  - Computer Programming for Beginners: This Book incl
  - C Programming: A Modern Approach, 2nd Edition
  - Starting Out with Programming Logic and Design (3r
  - Java™ Programming: From Problem Analysis to Progra

Hybrid (combined): 5 results
  [BM25] Parallel and Concurrent Programming in Haskell: Te
  [Semantic] The Science of Programming (Monographs in Computer
  [BM25] Beginning C for Arduino, Second Edition: Learn C P
  [Semantic] C Programming: A Modern Approach, 2nd Edition
  [BM25] The Art of LEGO MINDSTORMS EV3 Programming (Full C
  From BM25 only: 3
  From Semantic onl

## 7. Verify Hybrid Works

Final verification that hybrid retrieval is working correctly and ready for RAG.

**Output:** Status and readiness for next phase

In [8]:
print("Hybrid Retrieval Verification")
print("=" * 80)

test_count = 3
all_working = True

for i in range(test_count):
    try:
        results = hybrid_retriever.search(f"test query {i}", top_k=5)
        if len(results) > 0 and len(results) <= 5:
            print(f"Test {i+1}: PASS (got {len(results)} results)")
        else:
            print(f"Test {i+1}: FAIL (invalid result count)")
            all_working = False
    except Exception as e:
        print(f"Test {i+1}: ERROR - {str(e)}")
        all_working = False

print()
if all_working:
    print("VERIFICATION PASSED: Hybrid retrieval ready")
    print("Next: Run 07_rag_pipeline.ipynb")
else:
    print("VERIFICATION FAILED: Check errors above")

Hybrid Retrieval Verification
Test 1: PASS (got 5 results)
Test 2: PASS (got 5 results)
Test 3: PASS (got 5 results)

VERIFICATION PASSED: Hybrid retrieval ready
Next: Run 07_rag_pipeline.ipynb
